# Hardware Telemetry

Collects machine metadata for the configured run(s) and writes one `hardware/run_{run_index}.json` file per selected run under the active evaluation namespace.

## Environment

In [ ]:
import sys
from pathlib import Path

import yaml


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Evaluation Config

In [ ]:
def resolve_run_indices(n_runs: int, run_start: int = 1, run_count: int | None = None) -> list[int]:
    if n_runs <= 0:
        raise ValueError("n_runs must be positive")
    if run_start <= 0:
        raise ValueError("run_start must be positive")

    if run_count is None:
        run_end = n_runs
    else:
        run_count = int(run_count)
        if run_count <= 0:
            raise ValueError("run_count must be positive when set")
        run_end = run_start + run_count - 1

    if run_end > n_runs:
        raise ValueError(
            f"run_start/run_count select runs {run_start}..{run_end}, "
            f"but n_runs={n_runs}"
        )

    return list(range(run_start, run_end + 1))


EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

model_key = eval_config["model_key"]
eval_milestones = list(eval_config["eval_milestones"])
n_runs = int(eval_config["n_runs"])
run_start = int(eval_config.get("run_start", 1))
raw_run_count = eval_config.get("run_count")
run_count = None if raw_run_count is None else int(raw_run_count)
run_indices = resolve_run_indices(n_runs, run_start, run_count)
judge_repeats = int(eval_config["judge"]["repeats"])

print(f"hardware telemetry target: model={model_key} runs={run_indices} r{n_runs}-j{judge_repeats}")

## Manual Provider Metadata

In [ ]:
NOTES = ""

MANUAL_METADATA = {
    "provider_reported_dlperf": "",
    "provider_reported_tflops": "",
    "provider_reported_vram_bandwidth_gbps": "",
    "provider_reported_pcie": "",
    "provider_reported_pcie_bandwidth_gbps": "",
}
MANUAL_METADATA = {
    key: value
    for key, value in MANUAL_METADATA.items()
    if value not in (None, "")
}

## Capture

In [ ]:
import copy

from src.evaluate import collect_hardware_info, save_hardware_results


hardware_template = collect_hardware_info(
    run_index=None,
    notes=NOTES,
    manual=MANUAL_METADATA,
)

hardware_paths = []
hardware_by_run = {}
for run_index in run_indices:
    hardware = copy.deepcopy(hardware_template)
    hardware["run_index"] = run_index
    hardware_path = save_hardware_results(
        hardware=hardware,
        model_key=model_key,
        milestones=eval_milestones,
        run_index=run_index,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
    )
    hardware_paths.append(hardware_path)
    hardware_by_run[run_index] = hardware
    print(f"saved hardware telemetry: run={run_index}/{n_runs} path={hardware_path}")

hardware_by_run